# Annotator Analysis
### Finding Common Patterns in Harm Ratings

---

## Goal

We want to understand: **do annotators with similar characteristics rate prompts similarly?**

For each annotator feature (demographic and psychological), we test:
> "Do annotators with higher/different values on this feature rate prompts differently?"

We test this:
1. **Across all prompts** — general effect
2. **Only on highly contested prompts** — where disagreement is highest

**Statistical test:** Mann-Whitney U test (non-parametric, suitable for ratings)
- p < 0.05 → statistically significant difference
- We report mean rating difference between groups

**Even if we find nothing, that is a finding too.**
> "I tested X — it does not explain rating differences."


## 0. Setup

In [6]:
import sys, os, warnings
warnings.filterwarnings('ignore')

REPO_ROOT = r"C:\Users\hp\PluriHarms-release"
sys.path.insert(0, os.path.join(REPO_ROOT, 'src'))
os.chdir(REPO_ROOT)
os.makedirs("outputs", exist_ok=True)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu
from data_loader import load_data, get_rating_columns

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.dpi'] = 110

print("Setup complete.")

Setup complete.


## 1. Load Data

In [7]:
annotations, prompts = load_data()
rating_cols = get_rating_columns(annotations)

# Rating matrix
ratings_matrix = annotations[rating_cols].copy()
ratings_matrix.index = annotations['Participant_ID']

# Disagreement per prompt
std_per_prompt = {}
avg_per_prompt = {}
for col in rating_cols:
    q_idx = int(col.replace('Rating_', ''))
    std_per_prompt[q_idx] = annotations[col].std()
    avg_per_prompt[q_idx] = annotations[col].mean()

std_series = pd.Series(std_per_prompt)
median_std = std_series.median()
contested_prompts = std_series[std_series > median_std].index.tolist()

print(f"Annotators     : {annotations.shape[0]}")
print(f"Prompts        : {len(rating_cols)}")
print(f"Median std     : {median_std:.2f}")
print(f"Contested prompts (std > median): {len(contested_prompts)}")

Annotators     : 100
Prompts        : 150
Median std     : 29.61
Contested prompts (std > median): 75


## 2. Prepare Long Format

One row per annotator-prompt pair.
Each row has the rating + all annotator features.


In [8]:
cat_D_raw = ['Gender', 'Age', 'Political_Affiliation', 'Education',
             'Social_Media_Frequency', 'Toxicity_Experience',
             'Income', 'Religion_Importance',
             'Race_Ethnicity', 'Sexual_Orientation']
cat_E = ['Factor_1', 'Factor_2', 'Factor_3']

rows = []
for _, ann_row in annotations.iterrows():
    for col in rating_cols:
        q_idx  = int(col.replace('Rating_', ''))
        rating = ann_row[col]
        if pd.isna(rating):
            continue
        row = {
            'rating'         : rating,
            'question_index' : q_idx,
            'Participant_ID' : ann_row['Participant_ID'],
            'is_contested'   : q_idx in contested_prompts,
        }
        for c in cat_D_raw + cat_E:
            row[c] = ann_row[c]
        rows.append(row)

long_df = pd.DataFrame(rows)

print(f"Long format: {long_df.shape[0]:,} rows x {long_df.shape[1]} columns")
print(f"Contested rows   : {long_df['is_contested'].sum():,}")
print(f"Uncontested rows : {(~long_df['is_contested']).sum():,}")

Long format: 14,725 rows x 17 columns
Contested rows   : 7,305
Uncontested rows : 7,420


## 3. Test Categorical Features

For each categorical feature, we split annotators into groups
and test whether their ratings differ significantly.

**Method:** Mann-Whitney U test
- H0: the two groups have the same rating distribution
- p < 0.05: reject H0 → groups rate differently


In [9]:
def test_categorical(df, feature, group_a_label, group_b_label,
                     group_a_values, group_b_values):
    """
    Test whether two groups of annotators rate differently.
    Returns: mean_a, mean_b, difference, p_value, significant
    """
    group_a = df[df[feature].isin(group_a_values)]['rating']
    group_b = df[df[feature].isin(group_b_values)]['rating']

    if len(group_a) < 10 or len(group_b) < 10:
        return None

    stat, p = mannwhitneyu(group_a, group_b, alternative='two-sided')

    return {
        'feature'   : feature,
        'group_a'   : group_a_label,
        'group_b'   : group_b_label,
        'n_a'       : len(group_a),
        'n_b'       : len(group_b),
        'mean_a'    : round(group_a.mean(), 2),
        'mean_b'    : round(group_b.mean(), 2),
        'difference': round(group_b.mean() - group_a.mean(), 2),
        'p_value'   : round(p, 4),
        'significant': p < 0.05,
    }

print("test_categorical() function ready.")

test_categorical() function ready.


In [10]:
# Define group comparisons for categorical features
comparisons = [
    # (feature, group_a_label, group_b_label, group_a_values, group_b_values)
    ('Gender', 'Man', 'Woman',
     ['Man'], ['Woman']),

    ('Political_Affiliation', 'Liberal', 'Conservative',
     ['Very liberal', 'Somewhat liberal'],
     ['Very conservative', 'Somewhat conservative']),

    ('Race_Ethnicity', 'White', 'Non-white',
     ['White'],
     ['Black or African American', 'Latino or Hispanic', 'East Asian',
      'South Asian', 'Other', 'East Asian,South Asian',
      'Latino or Hispanic,White', 'Black or African American,White',
      'American Indian, Alaska Native, or']),

    ('Education', 'Low education', 'High education',
     ['High school diploma or GED', 'Some college, but no degree',
      'Associates or technical degree'],
     ["Bachelor’s degree",
      'Graduate or professional degree (MA, MS, MBA, PhD, JD, MD, DDS etc.)']),

    ('Age', 'Young (18-34)', 'Older (35+)',
     ['18-24 years old', '25-34 years old'],
     ['35-44 years old', '45-54 years old',
      '55-64 years old', '65 years old or above']),

    ('Sexual_Orientation', 'Straight', 'Non-straight',
     ['Straight'], ['Gay or lesbian', 'Bisexual', 'Other']),

    ('Religion_Importance', 'Not religious', 'Religious',
     ['Not important at all', 'Not very important'],
     ['Very important', 'Somewhat important']),

    ('Toxicity_Experience', 'No toxicity', 'Has toxicity experience',
     ['Never a problem'],
     ['Sometimes a problem', 'Often a problem', 'Very often a problem']),
]

print("Testing categorical features...")
print()

cat_results_all       = []
cat_results_contested = []

for comp in comparisons:
    feature, label_a, label_b, vals_a, vals_b = comp

    # All prompts
    res_all = test_categorical(long_df, feature, label_a, label_b, vals_a, vals_b)
    if res_all:
        cat_results_all.append(res_all)

    # Contested prompts only
    res_cont = test_categorical(
        long_df[long_df['is_contested']], feature, label_a, label_b, vals_a, vals_b
    )
    if res_cont:
        cat_results_contested.append(res_cont)

cat_all_df  = pd.DataFrame(cat_results_all)
cat_cont_df = pd.DataFrame(cat_results_contested)

print("ALL PROMPTS:")
print(cat_all_df[['feature','group_a','group_b','mean_a','mean_b',
                   'difference','p_value','significant']].to_string(index=False))
print()
print("CONTESTED PROMPTS ONLY:")
print(cat_cont_df[['feature','group_a','group_b','mean_a','mean_b',
                    'difference','p_value','significant']].to_string(index=False))

Testing categorical features...

ALL PROMPTS:
              feature       group_a        group_b  mean_a  mean_b  difference  p_value  significant
               Gender           Man          Woman   47.51   51.39        3.89   0.0000         True
Political_Affiliation       Liberal   Conservative   49.88   50.57        0.69   0.1495        False
       Race_Ethnicity         White      Non-white   49.80   49.84        0.04   0.2872        False
            Education Low education High education   51.17   48.02       -3.15   0.0000         True
                  Age Young (18-34)    Older (35+)   49.81   49.12       -0.69   0.2569        False

CONTESTED PROMPTS ONLY:
              feature       group_a        group_b  mean_a  mean_b  difference  p_value  significant
               Gender           Man          Woman   51.43   56.45        5.02   0.0000         True
Political_Affiliation       Liberal   Conservative   54.34   55.03        0.69   0.1770        False
       Race_Ethnicit

## 4. Test Psychological Features (Factor Scores)

In [11]:
def test_factor(df, factor_col, label_low, label_high):
    """
    Split annotators by median factor score → test rating difference.
    Low = below median, High = above median
    """
    median_val = df[factor_col].median()
    low_group  = df[df[factor_col] <= median_val]['rating']
    high_group = df[df[factor_col] >  median_val]['rating']

    if len(low_group) < 10 or len(high_group) < 10:
        return None

    stat, p = mannwhitneyu(low_group, high_group, alternative='two-sided')

    return {
        'feature'    : factor_col,
        'group_low'  : f'{label_low} (≤ median)',
        'group_high' : f'{label_high} (> median)',
        'n_low'      : len(low_group),
        'n_high'     : len(high_group),
        'mean_low'   : round(low_group.mean(), 2),
        'mean_high'  : round(high_group.mean(), 2),
        'difference' : round(high_group.mean() - low_group.mean(), 2),
        'p_value'    : round(p, 4),
        'significant': p < 0.05,
    }

factor_tests = [
    ('Factor_1', 'Universalist/Benevolent', 'Authority/Power'),
    ('Factor_2', 'Low empathy', 'High empathy'),
    ('Factor_3', 'Hedonism/Openness', 'Tradition/Conformity'),
]

print("Testing psychological features...")
print()

fac_results_all       = []
fac_results_contested = []

for factor_col, label_low, label_high in factor_tests:
    res_all = test_factor(long_df, factor_col, label_low, label_high)
    if res_all:
        fac_results_all.append(res_all)

    res_cont = test_factor(
        long_df[long_df['is_contested']], factor_col, label_low, label_high
    )
    if res_cont:
        fac_results_contested.append(res_cont)

fac_all_df  = pd.DataFrame(fac_results_all)
fac_cont_df = pd.DataFrame(fac_results_contested)

print("ALL PROMPTS:")
print(fac_all_df[['feature','mean_low','mean_high',
                   'difference','p_value','significant']].to_string(index=False))
print()
print("CONTESTED PROMPTS ONLY:")
print(fac_cont_df[['feature','mean_low','mean_high',
                    'difference','p_value','significant']].to_string(index=False))

Testing psychological features...

ALL PROMPTS:
 feature  mean_low  mean_high  difference  p_value  significant
Factor_1     50.53      48.18       -2.35   0.0000         True
Factor_2     49.06      49.67        0.61   0.9496        False
Factor_3     49.77      48.95       -0.82   0.1570        False

CONTESTED PROMPTS ONLY:
 feature  mean_low  mean_high  difference  p_value  significant
Factor_1     56.09      51.54       -4.55   0.0000         True
Factor_2     53.32      54.33        1.01   0.4389        False
Factor_3     54.69      52.94       -1.75   0.0803        False


## 5. Visualize Results

In [12]:
# Combine all results
all_results = []
for _, row in cat_all_df.iterrows():
    all_results.append({
        'Feature'    : f"{row['feature']}: {row['group_a']} vs {row['group_b']}",
        'Difference' : row['difference'],
        'p_value'    : row['p_value'],
        'Significant': row['significant'],
        'Type'       : 'Demographic'
    })
for _, row in fac_all_df.iterrows():
    all_results.append({
        'Feature'    : row['feature'],
        'Difference' : row['difference'],
        'p_value'    : row['p_value'],
        'Significant': row['significant'],
        'Type'       : 'Psychological'
    })

results_df = pd.DataFrame(all_results).sort_values('Difference')

fig, ax = plt.subplots(figsize=(12, 7))

colors = []
for _, row in results_df.iterrows():
    if row['Significant']:
        colors.append('#e74c3c' if row['Difference'] > 0 else '#3498db')
    else:
        colors.append('#bdc3c7')

bars = ax.barh(results_df['Feature'], results_df['Difference'],
               color=colors, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.bar_label(bars, labels=[f"{v:+.2f}" for v in results_df['Difference']],
             padding=3, fontsize=8)
ax.set_xlabel('Mean Rating Difference (Group B - Group A)')
ax.set_title('Rating Differences by Annotator Feature\n(All Prompts)',
             fontweight='bold', fontsize=12)

from matplotlib.patches import Patch
legend = [
    Patch(color='#e74c3c', label='Significant: Group B rates higher'),
    Patch(color='#3498db', label='Significant: Group B rates lower'),
    Patch(color='#bdc3c7', label='Not significant (p ≥ 0.05)'),
]
ax.legend(handles=legend, fontsize=9)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig("outputs/fig_annotator_differences.png", bbox_inches='tight')
plt.show()
print("Saved: outputs/fig_annotator_differences.png")

SyntaxError: unterminated string literal (detected at line 37) (1384004984.py, line 37)

In [ ]:
# Contested vs All comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, df_cat, df_fac, title in zip(
    axes,
    [cat_all_df, cat_cont_df],
    [fac_all_df, fac_cont_df],
    ['All Prompts', 'Contested Prompts Only']
):
    rows = []
    for _, row in df_cat.iterrows():
        rows.append({
            'Feature'    : f"{row['feature']}",
            'Difference' : row['difference'],
            'Significant': row['significant'],
        })
    for _, row in df_fac.iterrows():
        rows.append({
            'Feature'    : row['feature'],
            'Difference' : row['difference'],
            'Significant': row['significant'],
        })

    plot_df = pd.DataFrame(rows).sort_values('Difference')
    colors  = ['#e74c3c' if (s and d > 0) else '#3498db' if (s and d <= 0)
               else '#bdc3c7'
               for s, d in zip(plot_df['Significant'], plot_df['Difference'])]

    bars = ax.barh(plot_df['Feature'], plot_df['Difference'],
                   color=colors, edgecolor='white')
    ax.axvline(0, color='black', linewidth=0.8)
    ax.bar_label(bars, labels=[f"{v:+.2f}" for v in plot_df['Difference']],
                 padding=3, fontsize=8)
    ax.set_title(title, fontweight='bold', fontsize=11)
    ax.set_xlabel('Mean Rating Difference')
    ax.spines[['top','right']].set_visible(False)

fig.suptitle('Annotator Feature Effects — All vs Contested Prompts',
             fontweight='bold', fontsize=12)
plt.tight_layout()
plt.savefig("outputs/fig_all_vs_contested.png", bbox_inches='tight')
plt.show()
print("Saved: outputs/fig_all_vs_contested.png")

## 6. Findings

In [ ]:
print("=" * 60)
print("FINDINGS SUMMARY")
print("=" * 60)
print()

print("CATEGORICAL FEATURES — All Prompts:")
sig = cat_all_df[cat_all_df['significant']]
not_sig = cat_all_df[~cat_all_df['significant']]

if len(sig) > 0:
    for _, row in sig.iterrows():
        direction = "higher" if row['difference'] > 0 else "lower"
        print(f"  ✓ {row['feature']}: {row['group_b']} rates {direction} "
              f"by {abs(row['difference']):.2f} points (p={row['p_value']:.4f})")
else:
    print("  No significant differences found.")

print()
if len(not_sig) > 0:
    print("  Not significant:")
    for _, row in not_sig.iterrows():
        print(f"  ✗ {row['feature']}: difference={row['difference']:+.2f} (p={row['p_value']:.4f})")

print()
print("PSYCHOLOGICAL FEATURES — All Prompts:")
sig_fac = fac_all_df[fac_all_df['significant']]
not_sig_fac = fac_all_df[~fac_all_df['significant']]

if len(sig_fac) > 0:
    for _, row in sig_fac.iterrows():
        direction = "higher" if row['difference'] > 0 else "lower"
        print(f"  ✓ {row['feature']}: High group rates {direction} "
              f"by {abs(row['difference']):.2f} points (p={row['p_value']:.4f})")
else:
    print("  No significant differences found.")

print()
if len(not_sig_fac) > 0:
    print("  Not significant:")
    for _, row in not_sig_fac.iterrows():
        print(f"  ✗ {row['feature']}: difference={row['difference']:+.2f} (p={row['p_value']:.4f})")

print()
print("CONTESTED PROMPTS — Additional findings:")
sig_cont_cat = cat_cont_df[cat_cont_df['significant']]
sig_cont_fac = fac_cont_df[fac_cont_df['significant']]

if len(sig_cont_cat) > 0 or len(sig_cont_fac) > 0:
    for _, row in sig_cont_cat.iterrows():
        direction = "higher" if row['difference'] > 0 else "lower"
        print(f"  ✓ {row['feature']}: {row['group_b']} rates {direction} "
              f"by {abs(row['difference']):.2f} (p={row['p_value']:.4f}) — contested only")
    for _, row in sig_cont_fac.iterrows():
        direction = "higher" if row['difference'] > 0 else "lower"
        print(f"  ✓ {row['feature']}: High group rates {direction} "
              f"by {abs(row['difference']):.2f} (p={row['p_value']:.4f}) — contested only")
else:
    print("  No additional significant findings on contested prompts.")

## 7. How to Report These Findings

For each test, write:

**If significant:**
> "I tested [feature] — [group] rates prompts [higher/lower] by [X] points on average (p=[value]). This suggests that [interpretation]."

**If not significant:**
> "I tested [feature] — no significant difference in ratings was found (p=[value]). This suggests that [feature] alone does not predict rating behavior."

**Key principle from supervisor:**
> "Even if all hypotheses are wrong and we find no correlation, we need to defend by explaining our tests."
